In [1]:
import sys
import os
import numpy as np
from scipy import stats
import time
import pandas as pd

sys.path.append(os.path.abspath('..'))

from LSM.stochastic_processes import GeometricBrownianMotion
from LSM.payoffs import AmericanOption
from LSM.regression_bases import LaguerrePolynomials
from LSM.algorithms import LeastSquaresMonteCarlo

%load_ext autoreload
%autoreload 2

In [2]:
#(1) BSM Benchmark: Black-Scholes-Merton Model for European Options
def BSM(S0: float, K: float, T: float, r: float, q: float, sigma: float, option_type: str) -> float:
    """
    Returns the BSM price estimation of a European Option
    """
    option_type = option_type.lower()
    if option_type not in ['call', 'put']:
        raise ValueError("option_type must be either 'call' or 'put'.")

    d1 = (np.log(S0/K) + (r - q + 0.5*(sigma**2))*T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    if option_type == 'put':
        price = np.exp(-r*T) * K * stats.norm.cdf(-d2) - np.exp(-q*T) * S0 * stats.norm.cdf(-d1)
    else:
        price = np.exp(-q*T) * S0 * stats.norm.cdf(d1) - np.exp(-r*T) * K * stats.norm.cdf(d2)

    return price

In [3]:
n_paths = 10000  # Reduced from 100,000 for faster testing (still accurate)
n_steps = 50
S0 = 100
K = 100
T = 1.0
r = 0.05
q = 0.0
sigma = 0.2

start_bsm = time.time()
bsm_call = BSM(S0=S0, K=K, T=T, r=r, q=q, sigma=sigma, option_type="call")
end_bsm = time.time()
print(f"BSM European Call Price {bsm_call}.")


lsm_eng = LeastSquaresMonteCarlo(
    process=GeometricBrownianMotion(S0=S0, r=r, q=q, sigma=sigma), 
    payoff_function=AmericanOption(strike=K, option_type="call"),
    basis_function=LaguerrePolynomials(degree=3))
    

start_lsm = time.time()
lsm_call, lsm_std = lsm_eng.pricer(T = T, n_steps = n_steps, n_paths = n_paths)
end_lsm = time.time()
print(f"LSM American Call Price {lsm_call}.")
print(f"Absolute Error between BSM and LSM on Call: {abs(bsm_call - lsm_call)}")
print(f"LSM Standard Error: {lsm_std}")

print(f"BSM Runtime: {end_bsm - start_bsm:.6f}s")
print(f"LSM Runtime: {end_lsm - start_lsm:.6f}s")

BSM European Call Price 10.450583572185565.
LSM American Call Price 10.490070302791782.
Absolute Error between BSM and LSM on Call: 0.039486730606217435
LSM Standard Error: 0.1453398445809861
BSM Runtime: 0.002007s
LSM Runtime: 0.037172s


In [4]:
def LSM(S0, K, T, r, q, sigma, n_steps, n_paths, degree, option_type='put'):
    gbm = GeometricBrownianMotion(S0=S0, r=r, q=q, sigma=sigma)
    payoff = AmericanOption(strike=K, option_type=option_type)  # Now respects option_type!
    laguerre_basis = LaguerrePolynomials(degree=degree)  # Now respects degree!

    lsm_engine = LeastSquaresMonteCarlo(
        process=gbm, 
        payoff_function=payoff, 
        basis_function=laguerre_basis
    )

    price, _ = lsm_engine.pricer(T=T, n_steps=n_steps, n_paths=n_paths)

    return price

In [5]:
# Test Example Binomial Tree Benchmark:
def BTM(S0, K, T, r, q, sigma, n_steps, option_type = 'put'):
    return 2

In [6]:
# Test Example Finite Difference Benchmark: 
def FDM(S0, K, T, r, q, sigma, n_steps, n_paths, option_type = "put"):
    return 6


In [7]:
#(4) Comparison Table for LSM Accuracy and RunTime （需要验证）
# Assume here it is always American Put Option
# The Benchmark must be the estimated price by a highly stable and accurate model,
    #i.e., Binominal Tree Method

def comparison(model, name: str, benchmark_price, **kwargs):
    """
    Take all existed models (BSM, LSM, ...) to compute prices
    Compare Accuracy and RunTime of our LSM with other models
    Return a Comparison Table
    """

    start = time.time()
    price = model(**kwargs)
    end = time.time()

    runtime = end - start
    abs_error = abs(price - benchmark_price)
    #rel_error = abs_error / benchmark_price

    Ctable = {
        "Model": name,
        "Price": price,
        "Absolute Error": abs_error,
        "RunTime": runtime
    }

    return Ctable
 


In [8]:
#Parameters 
K=40
S0=36
sigma=0.2
T=2.0
r=0.06
q=0.0
n_paths=50000
n_steps=int(50 * T)

cargs = dict(S0=S0, K=K, T=T, r=r, q=q, sigma=sigma, option_type = 'put')

#Benchmark
benchmark_price = LSM(**cargs, n_steps=n_steps, n_paths=n_paths, degree=3)

#Realization of Comparison Table
models = []

models.append(
    comparison(LSM, "Least Square Monte Carlo", benchmark_price, **cargs, n_steps=n_steps, n_paths=n_paths, degree=3))
models.append(
    comparison(BTM, "Binomial Tree Model", benchmark_price, **cargs, n_steps=n_steps))
models.append(
    comparison(FDM, "Finite Difference Method", benchmark_price, **cargs, n_steps=n_steps, n_paths=n_paths))
models.append(
    comparison(BSM, "Black Scholes Model", benchmark_price, **cargs)
)

#Convert into a Data Frame
df = pd.DataFrame(models)

print(df)

                      Model     Price  Absolute Error   RunTime
0  Least Square Monte Carlo  4.857893        0.016830  0.645661
1       Binomial Tree Model  2.000000        2.874723  0.000000
2  Finite Difference Method  6.000000        1.125277  0.000000
3       Black Scholes Model  3.763001        1.111722  0.000000


In [9]:
# Reproduce Table 1 from Longstaff-Schwartz (2001)
# American Put option pricing with various parameters
# Table 1: K=40, r=0.06, exercise points = 50 per year
# Uses 100,000 paths (50,000 base + 50,000 antithetic) for variance reduction

import itertools

# Paper values from Table 1 (Simulated American column with s.e.)
paper_values = {
    (36, 0.20, 1): (4.472, 0.010),
    (36, 0.20, 2): (4.821, 0.012),
    (36, 0.40, 1): (7.091, 0.020),
    (36, 0.40, 2): (8.488, 0.024),
    (38, 0.20, 1): (3.244, 0.009),
    (38, 0.20, 2): (3.735, 0.011),
    (38, 0.40, 1): (6.139, 0.019),
    (38, 0.40, 2): (7.669, 0.022),
    (40, 0.20, 1): (2.313, 0.009),
    (40, 0.20, 2): (2.879, 0.010),
    (40, 0.40, 1): (5.308, 0.018),
    (40, 0.40, 2): (6.921, 0.022),
    (42, 0.20, 1): (1.617, 0.007),
    (42, 0.20, 2): (2.206, 0.010),
    (42, 0.40, 1): (4.588, 0.017),
    (42, 0.40, 2): (6.243, 0.021),
    (44, 0.20, 1): (1.118, 0.007),
    (44, 0.20, 2): (1.675, 0.009),
    (44, 0.40, 1): (3.957, 0.017),
    (44, 0.40, 2): (5.622, 0.021),
}

# Test parameters from Table 1
K = 40
r = 0.06
q = 0.0  # No dividend
spot_prices = [36, 38, 40, 42, 44]
volatilities = [0.20, 0.40]
times_to_maturity = [1, 2]
n_base_paths = 50000  # Will be doubled via antithetic
exercise_points_per_year = 50

# Generate all combinations
test_params = list(itertools.product(spot_prices, volatilities, times_to_maturity))

print("=" * 120)
print("Table 1: American Put Option Valuation (Longstaff-Schwartz 2001)")
print("=" * 120)
print(f"Strike Price K = {K}, Risk-free rate r = {r:.2%}, Dividend q = {q:.2%}")
print(f"Paths: {n_base_paths * 2:,} (base + antithetic), Exercise: 50 times per year")
print("-" * 120)
print(f"{'S0':<6} {'σ':<6} {'T':<4} {'Simulated Amer':<15} {'(s.e.)':<10} {'Paper Value':<15} {'Paper (s.e.)':<12} {'Error':<10}")
print("-" * 120)

results_lsm = []

for S0, sigma, T in test_params:
    n_steps = int(exercise_points_per_year * T)
    
    # Compute LSM American put price with antithetic variables
    gbm = GeometricBrownianMotion(S0=S0, r=r, q=q, sigma=sigma)
    put_payoff = AmericanOption(strike=K, option_type="put")
    laguerre_basis = LaguerrePolynomials(degree=3)
    
    lsm_engine = LeastSquaresMonteCarlo(
        process=gbm,
        payoff_function=put_payoff,
        basis_function=laguerre_basis
    )
    
    # Price with antithetic variables
    lsm_price, lsm_std = lsm_engine.pricer(
        T=T, 
        n_steps=n_steps, 
        n_paths=2 * n_base_paths,
        use_antithetic=True
    )
    
    # Get paper values
    paper_price, paper_std = paper_values[(S0, sigma, T)]
    error = abs(lsm_price - paper_price)
    error_pct = error / paper_price * 100
    
    print(f"{S0:<6} {sigma:<6.2f} {T:<4} {lsm_price:<15.4f} {f'({lsm_std:.4f})':<10} {paper_price:<15.4f} {f'({paper_std:.4f})':<12} {error_pct:<10.2f}%")
    
    results_lsm.append({
        'S0': S0,
        'sigma': sigma,
        'T': T,
        'LSM_Price': lsm_price,
        'LSM_Std': lsm_std,
        'Paper_Price': paper_price,
        'Paper_Std': paper_std,
        'Error': error,
        'Error_Pct': error_pct
    })

print("-" * 120)

# Summary statistics
df_results = pd.DataFrame(results_lsm)
mean_error = df_results['Error_Pct'].mean()
max_error = df_results['Error_Pct'].max()

print(f"\nSummary Statistics:")
print(f"  Mean absolute error: {mean_error:.2f}%")
print(f"  Max absolute error:  {max_error:.2f}%")
print(f"\nDetailed Results:")
print(df_results.to_string(index=False))

print("\n" + "=" * 120)
print("Interpretation:")
print("  - Errors < 1% indicate excellent reproduction of published benchmark")
print("  - Paper uses 100,000 paths (50,000 + 50,000 antithetic)")
print("  - Standard errors shown in parentheses")
print("=" * 120)

Table 1: American Put Option Valuation (Longstaff-Schwartz 2001)
Strike Price K = 40, Risk-free rate r = 6.00%, Dividend q = 0.00%
Paths: 100,000 (base + antithetic), Exercise: 50 times per year
------------------------------------------------------------------------------------------------------------------------
S0     σ      T    Simulated Amer  (s.e.)     Paper Value     Paper (s.e.) Error     
------------------------------------------------------------------------------------------------------------------------
36     0.20   1    4.4735          (0.0091)   4.4720          (0.0100)     0.03      %
36     0.20   2    4.8419          (0.0111)   4.8210          (0.0120)     0.43      %
36     0.40   1    7.1065          (0.0189)   7.0910          (0.0200)     0.22      %
36     0.40   2    8.5062          (0.0225)   8.4880          (0.0240)     0.21      %
38     0.20   1    3.2496          (0.0094)   3.2440          (0.0090)     0.17      %
38     0.20   2    3.7406          (0.0109

In [10]:
# Quick Demo: Variance Reduction via Antithetic Variables
# Fair comparison: same effective sample size (10,000) via different methods

print("\n" + "=" * 80)
print("Variance Reduction Demo: Antithetic Variables (Fair Comparison)")
print("=" * 80)

n_eff_paths = 10000  # Target effective sample size
n_steps_test = 50
S0_test, K_test, T_test = 100, 100, 1.0
r_test, q_test, sigma_test = 0.05, 0.0, 0.2

# Standard MC: 10,000 independent paths
gbm = GeometricBrownianMotion(S0=S0_test, r=r_test, q=q_test, sigma=sigma_test)
call_payoff = AmericanOption(strike=K_test, option_type="call")
laguerre_basis = LaguerrePolynomials(degree=3)

lsm_eng_no_anti = LeastSquaresMonteCarlo(
    process=gbm, payoff_function=call_payoff, basis_function=laguerre_basis
)

import time
start = time.time()
price_no_anti, std_no_anti = lsm_eng_no_anti.pricer(
    T=T_test, n_steps=n_steps_test, n_paths=n_eff_paths, use_antithetic=False
)
time_no_anti = time.time() - start

# MC with Antithetic: 10,000 base paths (generates 20,000 total after antithetic pairing)
lsm_eng_anti = LeastSquaresMonteCarlo(
    process=gbm, payoff_function=call_payoff, basis_function=laguerre_basis
)

start = time.time()
price_anti, std_anti = lsm_eng_anti.pricer(
    T=T_test, n_steps=n_steps_test, n_paths=n_eff_paths*2, use_antithetic=True
)
time_anti = time.time() - start

print(f"\nStandard MC:")
print(f"  Effective paths: {n_eff_paths:,} (all independent)")
print(f"  Price: {price_no_anti:.6f}, Std Error: {std_no_anti:.6f}")
print(f"  Runtime: {time_no_anti:.4f}s")

print(f"\nMC with Antithetic Variance Reduction:")
print(f"  Effective paths: {n_eff_paths*2:,} ({n_eff_paths:,} base + {n_eff_paths:,} antithetic pairs)")
print(f"  Price: {price_anti:.6f}, Std Error: {std_anti:.6f}")
print(f"  Runtime: {time_anti:.4f}s")

vrf = std_no_anti / std_anti
print(f"\nVariance Reduction Factor: {vrf:.2f}x")
print(f"Interpretation:")
print(f"  - Antithetic reduces std error by {(1 - 1/vrf)*100:.1f}%")
print(f"  - For same accuracy, antithetic needs only {(1/vrf**2)*100:.1f}% of the paths")
print(f"  - Speedup potential: {vrf**2:.2f}x (if all else equal)")
print("=" * 80)


Variance Reduction Demo: Antithetic Variables (Fair Comparison)

Standard MC:
  Effective paths: 10,000 (all independent)
  Price: 10.699208, Std Error: 0.145173
  Runtime: 0.0420s

MC with Antithetic Variance Reduction:
  Effective paths: 20,000 (10,000 base + 10,000 antithetic pairs)
  Price: 10.467877, Std Error: 0.104127
  Runtime: 0.0744s

Variance Reduction Factor: 1.39x
Interpretation:
  - Antithetic reduces std error by 28.3%
  - For same accuracy, antithetic needs only 51.4% of the paths
  - Speedup potential: 1.94x (if all else equal)


In [11]:
# Broadie-Glasserman (1997): American call on max of 5 assets (Table 6)
# Parameters: 5 assets, σ=20%, div=10%, r=5%, T=3, exercise 3x/year, K=100
# Paper reports: LSM prices [16.657, 26.182, 36.812] for S0 ∈ [90, 100, 110]

from LSM.stochastic_processes import GeometricBrownianMotion
from LSM.payoffs import MaxCallFeatures, AmericanMaxCall

print("\n" + "=" * 80)
print("Broadie-Glasserman (1997) Benchmark: American Call on Max of 5 Assets")
print("=" * 80)

# Parameters
n_assets = 5
K = 100
r = 0.05
q = 0.10  # 10% dividend
sigma = np.array([0.20] * n_assets)  # All 20%
T = 3.0
n_steps = 9  # 3x per year × 3 years
n_paths = 50000
correlation_matrix = np.eye(n_assets)  # Independent assets

test_S0_values = [90, 100, 110]
paper_values = [16.657, 26.182, 36.812]

results = []

for S0, paper_price in zip(test_S0_values, paper_values):
    S0_vector = np.array([S0] * n_assets)
    
    # Setup
    gbm = GeometricBrownianMotion(
        S0=S0_vector, r=r, q=np.array([q] * n_assets), 
        sigma=sigma, correlation_matrix=correlation_matrix
    )
    payoff = AmericanMaxCall(strike=K)
    features = MaxCallFeatures(strike=K)
    basis = LaguerrePolynomials(degree=3)
    
    lsm_eng = LeastSquaresMonteCarlo(process=gbm, payoff_function=payoff, basis_function=basis)
    
    # Price
    lsm_price, lsm_std = lsm_eng.pricer(T=T, n_steps=n_steps, n_paths=n_paths, create_features=features)
    
    error = abs(lsm_price - paper_price)
    
    print(f"\nS0 = {S0}:")
    print(f"  LSM Price:   {lsm_price:.3f}")
    print(f"  Paper Price: {paper_price:.3f}")
    print(f"  Error:       {error:.3f} ({error/paper_price*100:.1f}%)")
    print(f"  Std Error:   {lsm_std:.4f}")
    
    results.append({'S0': S0, 'LSM': lsm_price, 'Paper': paper_price, 'Error': error})

df_bg = pd.DataFrame(results)
print("\n" + df_bg.to_string(index=False))
print("=" * 80)


Broadie-Glasserman (1997) Benchmark: American Call on Max of 5 Assets

S0 = 90:
  LSM Price:   16.527
  Paper Price: 16.657
  Error:       0.130 (0.8%)
  Std Error:   0.0733

S0 = 100:
  LSM Price:   26.166
  Paper Price: 26.182
  Error:       0.016 (0.1%)
  Std Error:   0.0880

S0 = 110:
  LSM Price:   36.793
  Paper Price: 36.812
  Error:       0.019 (0.1%)
  Std Error:   0.1002

 S0       LSM  Paper    Error
 90 16.526730 16.657 0.130270
100 26.165859 26.182 0.016141
110 36.793386 36.812 0.018614
